# P9 - Đánh giá hallucination / groundedness cho KGQA (WebQSP, pool ReaRev)

Notebook này chạy **offline** trên records reader đã có từ P8 cell 08 (`results_gnn/b08_reader_*`, `b08g_reader_*`), không gọi lại LLM. Nó tách hai khái niệm:

- **Faithfulness**: câu trả lời có được graph evidence đã đưa cho reader hỗ trợ (entailed) hay không.
- **Correctness**: câu trả lời có đúng gold hay không.

Mỗi entity trong câu trả lời là một *claim*. Verifier hai tầng (`kgqa_bench/hallucination.py`):

| Tầng | Kiểm tra | Kết quả |
|---|---|---|
| 1 (deterministic) | entity là đích của path topic -> entity trong context đã chọn | `path_endpoint` |
| 1 | entity chỉ xuất hiện làm node trung gian | `in_context_no_proof` -> UNSUPPORTED |
| 1 | entity có trong subgraph nhưng không được retrieve | `in_subgraph_only` -> UNSUPPORTED |
| 1 | entity không có trong subgraph | `absent` -> UNSUPPORTED |
| 2 (semantic) | chuỗi quan hệ có hướng của path trùng reference proof (topic -> gold, chỉ dùng lúc đánh giá) hoặc entity là gold | SUPPORTED, ngược lại UNCERTAIN |
| 2 (tuỳ chọn) | LLM judge cho claim UNCERTAIN | SUPPORTED / CONTRADICTED / UNCERTAIN |

Nhãn cấp câu: `hallucinated` = có claim UNSUPPORTED hoặc CONTRADICTED; `hallucinated_strict` tính thêm UNCERTAIN. Abstention (`unknown`, `cannot determine`) không bị tính là hallucination.

Ba nhóm metric:

- **A. Faithfulness**: entity_unsupported_rate (proxy cũ), claim_hallucination_rate (CHR), question_hallucination_rate (QHR), graph_proof_support_rate, contradiction_rate, uncertain_rate.
- **B. Correctness**: llm_hit, llm_f1, grounded_correct_rate, grounded_answer_f1, correct_ungrounded_rate, abstention_rate, answer_coverage.
- **C. Risk**: đường risk-coverage xếp theo confidence của retriever cho hallucination và cho trả lời sai, AURC và risk tại các mức coverage.

Thứ tự chạy: 00 -> 01 -> 02 (bắt buộc), 03-09 phân tích, 10 judge (tuỳ chọn, cần GPU), 11 human review, 12 tổng kết.

In [ ]:
# ==== 00. Mount Drive + đường dẫn ====
from google.colab import drive; drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/ThS/Research/colab'
GRAPH_RAG  = f'{DRIVE_BASE}/graph_rag'
import os, sys, json, glob, shutil, time, subprocess
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'; sys.dont_write_bytecode = True
for d in glob.glob(f'{GRAPH_RAG}/**/__pycache__', recursive=True): shutil.rmtree(d, ignore_errors=True)
os.chdir(GRAPH_RAG); sys.path.insert(0, GRAPH_RAG)
!pip install -q datasets sentence-transformers transformers accelerate networkx tqdm matplotlib tabulate
import numpy as np, pandas as pd
from IPython.display import Image, display
from kgqa_bench import hallucination as H
from kgqa_bench import escalation as E
from kgqa_bench.data import load_spec
print('cwd =', os.getcwd(), '| hallucination module OK')

## 01 - Cấu hình: thư mục records, dataset spec, method, judge
`RESULTS` là thư mục kết quả của P8 (`results_gnn`). Mỗi thư mục `b08_reader_<model>` (prompt gốc) hoặc `b08g_reader_<model>` (prompt grounded), có hậu tố `_v3` nếu P8 chọn CGAR v3. `WEBQSP_TE` phải đúng spec đã dùng ở P8 để map qid về subgraph.

In [ ]:
RESULTS   = f'{GRAPH_RAG}/results_gnn'
CKPT      = f'{GRAPH_RAG}/ckpt'
WEBQSP_TE = f'gnnrag:{CKPT}/gnn_features_webqsp_test.pkl:webqsp:test'
METHODS   = ['gnnrag_all', 'gnnrag_top5', 'sbert_top5', 'cgar']   # thêm 'prism' nếu P8 đã chạy PRISM
REF       = 'gnnrag_all'
STRICT    = False          # True: UNCERTAIN cũng tính là hallucination (báo cả hai trong report)
JUDGE     = None           # ví dụ 'hf:Qwen/Qwen2.5-7B-Instruct' cho cell 10
REVIEW_N  = 200            # số dòng cho human review
READER_DIRS = sorted(d for d in glob.glob(f'{RESULTS}/b08*_reader_*') if os.path.exists(f'{d}/results.json'))
assert READER_DIRS, 'chưa có thư mục b08_reader_* với results.json -> chạy P8 cell 08 trước'
def tag_of(d):
    b = os.path.basename(d); prompt = 'grounded' if b.startswith('b08g') else 'chat'
    return b.split('_reader_')[-1], prompt
for d in READER_DIRS: print(os.path.basename(d), '->', tag_of(d))
# rescore offline (idempotent): parser D2 + abstention tách khỏi unsupported_rate, để llm_hit trong records khớp với P8 cell 08
from kgqa_bench import rescore as RS
for d in READER_DIRS: RS.rescore_dir(d, verbose=False)
ds = load_spec(WEBQSP_TE); print('dataset:', ds.key, len(ds), 'câu')

## 02 - Chạy verifier cho mọi reader / prompt
Mỗi thư mục sinh `hallucination/`: `hallucination.json`, `report_webqsp_test.md`, `claims_webqsp_test_<method>.jsonl`, `risk_coverage_webqsp_test.png`, `review_sample_webqsp_test.csv`.

In [ ]:
RES = {}
for d in READER_DIRS:
    t0 = time.time()
    RES[tag_of(d)] = H.run(d, [WEBQSP_TE], methods=METHODS, ref=REF, judge_spec=None, strict=STRICT,
                           review_n=REVIEW_N, n_bootstrap=10000, verbose=False)
    print(f'{os.path.basename(d)}: xong ({(time.time()-t0)/60:.1f} phút)')
def summ(res):   # DataFrame summaries của dataset đầu tiên
    dsk = list(res['datasets'])[0]; return pd.DataFrame(res['datasets'][dsk]['summaries']).T
A_COLS = ['n_answered','answer_coverage','abstention_rate','claims_per_answer','entity_unsupported_rate','claim_hallucination_rate',
          'question_hallucination_rate','question_hallucination_rate_strict','graph_proof_support_rate','uncertain_rate','contradiction_rate']
B_COLS = ['llm_hit','llm_f1','grounded_correct_rate','grounded_answer_f1','correct_ungrounded_rate','abstention_rate']
C_COLS = ['aurc_hallucination','aurc_incorrect','aurc_not_grounded_correct']
for key, res in RES.items():
    S = summ(res)
    print('\n#####', key, '| A. Faithfulness'); display(S[[c for c in A_COLS if c in S.columns]].astype(float).round(3))
    print('B. Correctness'); display(S[[c for c in B_COLS if c in S.columns]].astype(float).round(3))
    print('C. Risk (AURC, thấp hơn tốt hơn)'); display(S[[c for c in C_COLS if c in S.columns]].astype(float).round(3))

## 03 - Nhãn claim theo evidence type: proxy cũ sai ở đâu?
So sánh `legacy_in_context` (entity có xuất hiện trong context, proxy cũ) với nhãn mới. Hai ô đáng chú ý: claim **in_context_no_proof** (proxy cũ gọi là supported, verifier gọi là UNSUPPORTED - ca "Alice works_at X, X located_in London") và claim **path_endpoint nhưng UNCERTAIN** (có path tới entity nhưng quan hệ khác proof).

In [ ]:
def load_claims(d, method):
    p = f'{d}/hallucination/claims_webqsp_test_{method}.jsonl'
    return [json.loads(l) for l in open(p)] if os.path.exists(p) else []
CL = {}
for d in READER_DIRS:
    for m in METHODS:
        vs = load_claims(d, m)
        if vs: CL[(tag_of(d), m)] = vs
rows = []
for (key, m), vs in CL.items():
    for v in vs:
        for cl in v['claims']:
            rows.append(dict(reader=key[0], prompt=key[1], method=m, evidence_type=cl['evidence_type'], label=cl['label'],
                             legacy_in_context=cl['legacy_in_context'], is_gold=cl['is_gold']))
CLDF = pd.DataFrame(rows)
print('claims:', len(CLDF))
display(CLDF.groupby(['prompt','method','evidence_type','label']).size().unstack(fill_value=0))
print('\nProxy cũ (entity trong context) vs nhãn mới:')
display(pd.crosstab([CLDF['prompt'], CLDF['method']], [CLDF['legacy_in_context'], CLDF['label']]))
fn = CLDF[(CLDF.legacy_in_context) & (CLDF.label == 'UNSUPPORTED')]
fp = CLDF[(~CLDF.legacy_in_context) & (CLDF.label == 'SUPPORTED')]
print(f'\nproxy cũ bỏ sót (trong context nhưng UNSUPPORTED): {len(fn)} claim | proxy cũ báo oan (ngoài context theo substring nhưng SUPPORTED qua alias): {len(fp)} claim')

## 04 - Ví dụ cụ thể cho từng loại nhãn
In 3 ví dụ mỗi loại để kiểm tra bằng mắt trước khi tin số. Đọc `why` để biết verifier quyết định dựa trên gì.

In [ ]:
by_q = {q.qid: q for q in ds}
def show_examples(vs, pred, n=3, title=''):
    print('=' * 100); print(title); k = 0
    for v in vs:
        hits = [cl for cl in v['claims'] if pred(cl)]
        if not hits: continue
        q = by_q.get(v['qid']); k += 1
        print(f"\n[{v['qid']}] Q: {q.question if q else '?'}\n   gold: {q.answers[:5] if q else '?'}\n   pred: {v['llm_pred'][:200]!r}")
        for cl in hits[:2]: print(f"   claim={cl['claim']!r} label={cl['label']} evidence={cl['evidence_type']} gold={cl['is_gold']} | {cl['why']}")
        if k >= n: break
key0 = next(iter(CL)); print('reader/prompt/method =', key0)
vs0 = CL[key0]
show_examples(vs0, lambda c: c['evidence_type'] == 'in_context_no_proof', title='UNSUPPORTED: entity trong context nhưng không có path topic->entity (wrong-relation / reasoning error)')
show_examples(vs0, lambda c: c['evidence_type'] == 'absent', title='UNSUPPORTED: entity không có trong subgraph (parametric / fabricated)')
show_examples(vs0, lambda c: c['evidence_type'] == 'in_subgraph_only', title='UNSUPPORTED: entity có trong KG nhưng retrieval không lấy (retrieval gap)')
show_examples(vs0, lambda c: c['label'] == 'UNCERTAIN', title='UNCERTAIN: có path tới entity nhưng chuỗi quan hệ khác reference proof')
show_examples(vs0, lambda c: c['label'] == 'SUPPORTED' and not c['is_gold'], title='SUPPORTED nhưng không phải gold (gold thiếu / alias / multi-answer)')

## 05 - Error taxonomy và so sánh từng qid giữa GNN-RAG-all và CGAR
Bốn câu hỏi của §6 trong kế hoạch: (1) cả hai đủ evidence nhưng chỉ CGAR hallucinate; (2) all có path hỗ trợ mà CGAR bỏ; (3) cả hai thiếu evidence nhưng all vẫn đúng nhờ parametric knowledge; (4) CGAR confidence cao mà hallucinate.

In [ ]:
for key, res in RES.items():
    S = summ(res); print('\n#####', key, '| error taxonomy (số câu)')
    display(pd.DataFrame({m: S.loc[m, 'error_taxonomy'] for m in S.index}).fillna(0).astype(int))
def per_qid(key, a='cgar', b='gnnrag_all'):
    A = {v['qid']: v for v in CL[(key, a)]}; B = {v['qid']: v for v in CL[(key, b)]}
    q = sorted(set(A) & set(B)); out = {}
    out['(1) cả hai hit_ctx=1, chỉ CGAR hallucinate'] = [x for x in q if A[x]['hit_ctx'] and B[x]['hit_ctx'] and A[x]['hallucinated'] and not B[x]['hallucinated']]
    out['(2) all có proof (grounded_correct) mà CGAR không'] = [x for x in q if B[x]['grounded_correct'] and not A[x]['grounded_correct']]
    out['(3) cả hai hit_ctx=0 nhưng all llm_hit=1 (parametric)'] = [x for x in q if not A[x]['hit_ctx'] and not B[x]['hit_ctx'] and B[x]['llm_hit']]
    confs = [A[x]['confidence'] for x in q if A[x].get('confidence') is not None]; thr = np.quantile(confs, 0.8) if confs else 1.0
    out['(4) CGAR conf top-20% mà hallucinate'] = [x for x in q if A[x].get('confidence') is not None and A[x]['confidence'] >= thr and A[x]['hallucinated']]
    out['(5) CGAR grounded_correct, all hallucinate'] = [x for x in q if A[x]['grounded_correct'] and B[x]['hallucinated']]
    return out, A, B
for key in RES:
    if (key, 'cgar') in CL and (key, 'gnnrag_all') in CL:
        out, A, B = per_qid(key); print('\n#####', key)
        for k, v in out.items(): print(f'  {k}: {len(v)} câu')
        for x in out['(1) cả hai hit_ctx=1, chỉ CGAR hallucinate'][:3]:
            print(f"   e.g. {x}: CGAR pred={A[x]['llm_pred'][:80]!r} | all pred={B[x]['llm_pred'][:80]!r}")

## 06 - Risk-coverage: confidence của retriever có xếp hạng được hallucination không?
Đường risk-coverage: xếp câu theo confidence giảm dần, trục x = tỉ lệ câu được trả lời, trục y = tỉ lệ hallucinate (hoặc sai) trong số đó. Một method có thể hallucinate nhiều hơn ở coverage 100% nhưng an toàn hơn khi chỉ trả lời câu tự tin. Phải đọc kèm correctness tại cùng coverage để tránh thắng nhờ abstain.

In [ ]:
for key, res in RES.items():
    dsk = list(res['datasets'])[0]; S = res['datasets'][dsk]['summaries']
    rows = []
    for m, s in S.items():
        if 'halluc_risk_at' not in s: continue
        r = dict(method=m, aurc_halluc=s['aurc_hallucination'], aurc_incorrect=s['aurc_incorrect'])
        for c in ('0.2','0.5','0.8','1.0'): r[f'halluc@{c}'] = s['halluc_risk_at'][c]; r[f'incorrect@{c}'] = s['incorrect_risk_at'][c]
        rows.append(r)
    print('\n#####', key); display(pd.DataFrame(rows).set_index('method').round(3))
    d = [x for x in READER_DIRS if tag_of(x) == key][0]
    display(Image(f'{d}/hallucination/risk_coverage_webqsp_test.png'))
    if 'cgar' in S and REF in S and 'halluc_risk_at' in S['cgar']:
        c, a = S['cgar'], S[REF]
        print(f"CGAR + risk gate @50%: halluc {c['halluc_risk_at']['0.5']:.3f} vs all @100% {a['halluc_risk_at']['1.0']:.3f} | incorrect {c['incorrect_risk_at']['0.5']:.3f} vs {a['incorrect_risk_at']['1.0']:.3f}")

## 07 - Kiểm định cặp (paired bootstrap + McNemar) trên nhãn cấp câu
Δ âm ở `hallucinated` nghĩa là ít hallucination hơn tham chiếu. `grounded_correct` và `llm_hit` để chắc rằng giảm hallucination không phải do trả lời ít đi.

In [ ]:
for key, res in RES.items():
    dsk = list(res['datasets'])[0]; C = res['datasets'][dsk]['comparisons']; rows = []
    for m, byk in C.items():
        for k, c in byk.items():
            if not isinstance(c, dict) or 'bootstrap' not in c: continue
            b = c['bootstrap']
            rows.append(dict(method=m, key=k, n=c['n'], delta_pp=round(b['delta']*100, 2), ci=f"[{b['ci_low']*100:+.2f}, {b['ci_high']*100:+.2f}]",
                             p_boot=round(b['p_value'], 4), p_mcnemar=round(c['mcnemar']['p_value'], 4), sig=c['significant_0.05']))
    print('\n#####', key, '| A − ' + REF); display(pd.DataFrame(rows))
    for m, byk in C.items(): print(m, 'bất đồng hallucinated (a=method, b=ref):', byk.get('disagreement_hallucinated'))

## 08 - Prompt gốc vs prompt grounded (D1)
Cùng reader, cùng 1000 qid. Kỳ vọng: grounded giảm claim-HR và tăng abstention; cần xem `grounded_correct_rate` có giảm theo không.

In [ ]:
readers = sorted({k[0] for k in RES})
for rd in readers:
    if (rd, 'chat') in RES and (rd, 'grounded') in RES:
        a, g = summ(RES[(rd, 'chat')]), summ(RES[(rd, 'grounded')])
        cols = ['answer_coverage','abstention_rate','claim_hallucination_rate','question_hallucination_rate','llm_hit','grounded_correct_rate','grounded_answer_f1']
        cmp = pd.concat({'chat': a[cols].astype(float), 'grounded': g[cols].astype(float)}, axis=1).round(3)
        print('\n#####', rd); display(cmp)
        for m in ('gnnrag_all', 'cgar'):
            if m in a.index and m in g.index:
                print(f"  {m}: QHR {a.loc[m,'question_hallucination_rate']:.3f} -> {g.loc[m,'question_hallucination_rate']:.3f} | grounded_correct {a.loc[m,'grounded_correct_rate']:.3f} -> {g.loc[m,'grounded_correct_rate']:.3f} | abstain {a.loc[m,'abstention_rate']:.3f} -> {g.loc[m,'abstention_rate']:.3f}")
    else:
        print(rd, ': chỉ có một prompt -> bỏ qua so sánh')

## 09 - Escalation theo confidence ở mức hallucination
Chính sách hai tầng: CGAR nếu confidence >= τ, ngược lại dùng toàn bộ path của GNN-RAG-all. Tính trên verdict của hai method (cùng qid): QHR, grounded_correct, llm_hit và cost theo τ. Cho biết abstain/escalate có giảm hallucination mà không mất correctness không.

In [ ]:
def halluc_escalation(key, a='cgar', b='gnnrag_all'):
    A = {v['qid']: v for v in CL[(key, a)]}; B = {v['qid']: v for v in CL[(key, b)]}
    q = [x for x in sorted(set(A) & set(B)) if A[x].get('confidence') is not None]
    conf = np.array([A[x]['confidence'] for x in q]); taus = [-np.inf] + list(np.quantile(conf, np.linspace(0.1, 0.9, 9))) + [np.inf]
    rows = []
    for t in taus:
        use_b = conf < t; mixed = [B[x] if u else A[x] for x, u in zip(q, use_b)]
        rows.append(dict(tau=('-inf' if t == -np.inf else '+inf' if t == np.inf else round(float(t), 3)), escalated=float(use_b.mean()),
                         cost=np.mean([v['cost'] for v in mixed]), QHR=np.mean([v['hallucinated'] for v in mixed if v['answered']]),
                         grounded_correct=np.mean([v['grounded_correct'] for v in mixed]), llm_hit=np.mean([v['llm_hit'] for v in mixed])))
    return pd.DataFrame(rows)
for key in RES:
    if (key, 'cgar') in CL and (key, 'gnnrag_all') in CL:
        print('\n#####', key); display(halluc_escalation(key).round(4))

## 10 - (Tuỳ chọn, GPU) LLM judge cho claim UNCERTAIN
Judge chỉ xử lý claim UNCERTAIN và `in_context_no_proof`; trả về SUPPORTED / CONTRADICTED / UNCERTAIN. Đặt `JUDGE = 'hf:<model>'` ở cell 01. Không dùng một judge làm ground truth; dùng để thu hẹp vùng UNCERTAIN rồi đối chiếu với human review ở cell 11.

In [ ]:
if JUDGE:
    d = READER_DIRS[0]; key = tag_of(d)
    RESJ = H.run(d, [WEBQSP_TE], methods=METHODS, ref=REF, judge_spec=JUDGE, strict=STRICT, review_n=0, n_bootstrap=2000,
                 out_dir=f'{d}/hallucination_judge', verbose=False)
    base, judged = summ(RES[key]), summ(RESJ)
    cols = ['claim_hallucination_rate','question_hallucination_rate','uncertain_rate','contradiction_rate','grounded_correct_rate']
    display(pd.concat({'no_judge': base[cols].astype(float), 'judge': judged[cols].astype(float)}, axis=1).round(3))
else:
    print('[skip] JUDGE = None')

## 11 - Human review: xuất mẫu, nạp nhãn, đo detector
`review_sample_webqsp_test.csv` (cell 02) có cột `human_hallucinated` (0/1) và `human_label_per_claim` để người gán nhãn điền; nếu hai người gán, lưu hai file `..._A.csv`, `..._B.csv`. Cell này đo precision / recall / F1 của detector so với người và Cohen's kappa giữa hai người. Chỉ khi detector đạt mức chấp nhận được (ví dụ F1 >= 0.8) mới dùng số ở cell 02 để viết claim.

In [ ]:
def load_review(path):
    df = pd.read_csv(path); df = df[df['human_hallucinated'].astype(str).str.strip().isin(['0','1'])]
    df['human_hallucinated'] = df['human_hallucinated'].astype(int); return df
def detector_scores(df):
    y, p = df['human_hallucinated'].values, df['auto_hallucinated'].astype(int).values
    tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum()); fn = int(((y == 1) & (p == 0)).sum()); tn = int(((y == 0) & (p == 0)).sum())
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1); f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    return dict(n=len(df), tp=tp, fp=fp, fn=fn, tn=tn, precision=round(prec, 3), recall=round(rec, 3), f1=round(f1, 3), acc=round((tp + tn) / max(len(df), 1), 3))
def cohen_kappa(a, b):
    a, b = np.asarray(a), np.asarray(b); po = (a == b).mean()
    pe = sum(((a == k).mean()) * ((b == k).mean()) for k in set(a) | set(b)); return (po - pe) / max(1 - pe, 1e-9)
for d in READER_DIRS:
    base = f'{d}/hallucination/review_sample_webqsp_test'
    files = [p for p in (base + '.csv', base + '_A.csv', base + '_B.csv') if os.path.exists(p)]
    filled = []
    for p in files:
        df = load_review(p)
        if len(df): filled.append((p, df)); print(os.path.basename(d), os.path.basename(p), 'nhãn người:', len(df), '| detector:', detector_scores(df))
    if len(filled) >= 2:
        A, B = filled[-2][1].set_index(['qid','method']), filled[-1][1].set_index(['qid','method']); common = A.index.intersection(B.index)
        print('  Cohen kappa giữa hai người gán nhãn (n=%d): %.3f' % (len(common), cohen_kappa(A.loc[common,'human_hallucinated'], B.loc[common,'human_hallucinated'])))
    if not filled: print(os.path.basename(d), ': chưa có nhãn người trong', os.path.basename(base) + '.csv')

## 12 - Tổng kết: claim nào đứng được
Checklist tự động từ các bảng trên. Mỗi claim chỉ được viết vào báo cáo kèm Δ, CI, p của cell 07 và kết quả kiểm định detector ở cell 11.

In [ ]:
def gate(cond, msg): print(('PASS  ' if cond else 'FAIL  ') + msg)
lines = []
for key, res in RES.items():
    dsk = list(res['datasets'])[0]; S = res['datasets'][dsk]['summaries']; C = res['datasets'][dsk]['comparisons']
    if 'cgar' not in S or REF not in S: continue
    c = C.get('cgar', {}).get('hallucinated', {}).get('bootstrap', {}); g = C.get('cgar', {}).get('grounded_correct', {}).get('bootstrap', {})
    print('\n#####', key)
    gate(bool(c) and (c['delta'] <= 0 or c['p_value'] >= 0.05), f"CGAR không tăng question-HR so với {REF} (Δ={c.get('delta', 0)*100:+.2f} pp, p={c.get('p_value', 1):.3f})")
    gate(bool(g) and (g['delta'] >= 0 or g['p_value'] >= 0.05), f"CGAR không giảm grounded-correct so với {REF} (Δ={g.get('delta', 0)*100:+.2f} pp, p={g.get('p_value', 1):.3f})")
    gate('aurc_hallucination' in S['cgar'] and S['cgar']['aurc_hallucination'] < S[REF].get('aurc_hallucination', 1), f"AURC-hallucination CGAR {S['cgar'].get('aurc_hallucination')} < {REF} {S[REF].get('aurc_hallucination')}")
    gate(S['cgar']['halluc_risk_at']['0.5'] < S[REF]['halluc_risk_at']['1.0'] if 'halluc_risk_at' in S['cgar'] else False, 'CGAR + risk gate @50% coverage hallucinate ít hơn all @100%')
    gate(S['cgar']['abstention_rate'] <= S[REF]['abstention_rate'] + 0.05, 'CGAR không thắng nhờ abstain nhiều hơn (abstention chênh <= 5 pp)')
    lines.append(f"{key}: QHR cgar {S['cgar']['question_hallucination_rate']:.3f} vs {REF} {S[REF]['question_hallucination_rate']:.3f}; grounded-correct {S['cgar']['grounded_correct_rate']:.3f} vs {S[REF]['grounded_correct_rate']:.3f}")
summary_path = f'{RESULTS}/p9_hallucination_summary.md'
with open(summary_path, 'w') as f:
    f.write('# P9 hallucination summary\n\n' + '\n'.join('- ' + l for l in lines) + '\n\nChi tiết: <reader_dir>/hallucination/report_webqsp_test.md\n')
print('\n'.join(lines)); print('saved', summary_path)